## Good-Pose Feature Analysis (2Å)

Companion to `threshold_2.ipynb`, mirroring `good_pose_analysis_4A.ipynb` but at the
stricter 2Å threshold. Checking whether good-pose features generalize better here than
at 4Å (smaller good-pose class: 14.7% of test vs. 36.6% at 4Å) or worse (less signal to
learn per-feature correlations from, same instability pattern we already saw when the
bad-pose top-Spearman features got noisier going from 4Å to 2Å).

Reuses `threshold_2.ipynb`'s data loading / model loading / feature extraction /
correlation setup verbatim (sections 1-5), then the same good-pose-direction analysis as
`good_pose_analysis_4A.ipynb`.

**Output files (in `output/2A/`, same family as `threshold_2.ipynb`):**
- `good_pose_features_summary.csv` — per-feature precision/recall/F1 for the good-pose direction
- `threshold_sweep_good_k{k}.png` — activation distributions + threshold sweep
- `feature_activations_good_top5.csv` — exact case_id/sample_idx where these features fire


In [1]:
import warnings
import torch
import numpy as np
import pandas as pd
import sys
import os
from scipy.stats import spearmanr, ConstantInputWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=ConstantInputWarning)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_processor import load_processed_data
from src.model import TopKSAE

DATA_DIR       = '/Users/bridget/Desktop/projects/laloo-sae/processed_data'
OUT_DIR        = os.path.join(PROJECT_ROOT, 'output', '2A')
os.makedirs(OUT_DIR, exist_ok=True)
MODEL_DIR      = os.path.expanduser('~/Desktop/projects/laloo-sae/models/07_30_26')
RMSD_THRESHOLD = 2.0
K_VALUES       = [3, 8, 15]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Data dir:   {DATA_DIR}')
print(f'Output dir: {OUT_DIR}')
print(f'Model dir:  {MODEL_DIR}')

Device: cpu
Data dir:   /Users/bridget/Desktop/projects/laloo-sae/processed_data
Output dir: /Users/bridget/Desktop/projects/laloo-sae/output/2A
Model dir:  /Users/bridget/Desktop/projects/laloo-sae/models/07_30_26


### 1. Load data and update quality labels

In [2]:
latents_normalized, metadata, stats = load_processed_data(DATA_DIR)
print(f'Latents shape: {latents_normalized.shape}')
print(f'Metadata shape: {metadata.shape}')
metadata.head()

Loaded 435,069 samples from /Users/bridget/Desktop/projects/laloo-sae/processed_data
Latents shape: (435069, 30)
Metadata shape: (435069, 6)


,case_id,sample_idx,rmsd,energy,generation,global_idx
0,afab_2nnq_afab_3fr4,0,2.385954,-5425.862544,15,0
1,afab_2nnq_afab_3fr4,1,3.430689,10000.000000,15,1
2,afab_2nnq_afab_3fr4,2,3.147245,-5404.352427,8,2
3,afab_2nnq_afab_3fr4,3,2.383592,-5421.452032,14,3
4,afab_2nnq_afab_3fr4,4,3.503210,-5409.776425,8,4


In [3]:
metadata_4 = metadata.copy()
metadata_4['good_pose'] = metadata_4['rmsd'] < RMSD_THRESHOLD

n_good = metadata_4['good_pose'].sum()
n_bad  = (~metadata_4['good_pose']).sum()
print(f'RMSD threshold: {RMSD_THRESHOLD}Å')
print(f'  Good poses (RMSD < {RMSD_THRESHOLD}Å): {n_good:,} ({n_good/len(metadata_4)*100:.1f}%)')
print(f'  Bad  poses (RMSD >= {RMSD_THRESHOLD}Å): {n_bad:,} ({n_bad/len(metadata_4)*100:.1f}%)')

out_path = os.path.join(OUT_DIR, 'metadata.csv')
metadata_4.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

RMSD threshold: 2.0Å
  Good poses (RMSD < 2.0Å): 53,495 (12.3%)
  Bad  poses (RMSD >= 2.0Å): 381,574 (87.7%)



Saved: /Users/bridget/Desktop/projects/laloo-sae/output/2A/metadata.csv


### 2. Load train / test splits

In [4]:
splits    = np.load(os.path.join(DATA_DIR, 'splits.npz'), allow_pickle=True)
train_idx = splits['train_idx']
test_idx  = splits['test_idx']
print(f'Train: {len(train_idx):,}  |  Test: {len(test_idx):,}')

# Binary labels: 1 = bad pose, 0 = good pose
y_train = (~metadata_4['good_pose'].values[train_idx]).astype(int)
y_test  = (~metadata_4['good_pose'].values[test_idx]).astype(int)

# Raw latents (30-dim) -- baseline for logistic regression
X_raw_train = latents_normalized[train_idx]
X_raw_test  = latents_normalized[test_idx]

# Metadata for test-set evaluation (held out — never used to pick features/thresholds)
metadata_test = metadata_4.iloc[test_idx].reset_index(drop=True)
rmsd_test     = metadata_test['rmsd'].values
energy_test   = metadata_test['energy'].values
good_mask     = metadata_test['good_pose'].values

# Metadata for train-set feature/threshold selection (section 5/7 select on this, not on test)
metadata_train  = metadata_4.iloc[train_idx].reset_index(drop=True)
rmsd_train      = metadata_train['rmsd'].values
energy_train    = metadata_train['energy'].values
good_mask_train = metadata_train['good_pose'].values

print(f'Test good: {good_mask.sum():,} ({good_mask.mean()*100:.1f}%)')
print(f'Test bad:  {(~good_mask).sum():,} ({(~good_mask).mean()*100:.1f}%)')

Train: 294,851  |  Test: 68,614
Test good: 10,069 (14.7%)
Test bad:  58,545 (85.3%)


### 3. Load SAE models (k = 3, 8, 15)

In [5]:
summary = torch.load(
    os.path.join(MODEL_DIR, 'training_summary.pkl'),
    map_location='cpu', weights_only=False
)

trained_models = {}
for k in K_VALUES:
    run_stats    = summary[k]
    best_run_idx = int(np.argmin([r['best_val_loss'] for r in run_stats]))
    filepath     = os.path.join(MODEL_DIR, f'topksae_k{k}_run{best_run_idx}.pt')
    model = TopKSAE(input_dim=30, hidden_dim=120, k=k, auxk=12,
                    batch_size=256, dead_steps_threshold=2000).to(device)
    model.load_state_dict(
        torch.load(filepath, map_location=device, weights_only=False)
    )
    model.eval()
    trained_models[k] = model
    print(f'  [+] k={k}: run {best_run_idx}  '
          f'(val_loss={run_stats[best_run_idx]["best_val_loss"]:.4f})')

print(f'\nLoaded {len(trained_models)} models.')

  [+] k=3: run 2  (val_loss=0.3882)
  [+] k=8: run 0  (val_loss=0.2088)
  [+] k=15: run 3  (val_loss=0.0535)

Loaded 3 models.


### 4. Extract features (test + train) and save

In [6]:
BATCH_SIZE = 2048

def extract_activations(model, latents_np, batch_size=BATCH_SIZE):
    """Run model.get_acts() in batches. Input: numpy [N, 30]. Returns numpy [N, 120]."""
    all_acts = []
    for start in range(0, latents_np.shape[0], batch_size):
        batch = torch.tensor(latents_np[start:start + batch_size],
                             dtype=torch.float32, device=device)
        all_acts.append(model.get_acts(batch).cpu().numpy())
    return np.vstack(all_acts)


features_test  = {}   # correlation analysis + logistic regression eval
features_train = {}   # logistic regression fitting

for k in K_VALUES:
    print(f'k={k}:', end=' ', flush=True)
    features_test[k]  = extract_activations(trained_models[k], X_raw_test)
    print(f'test={features_test[k].shape}', end='  ', flush=True)
    features_train[k] = extract_activations(trained_models[k], X_raw_train)
    print(f'train={features_train[k].shape}  '
          f'zeros={(features_test[k]==0).mean()*100:.1f}%')

feat_path = os.path.join(OUT_DIR, 'features_for_paper.npz')
np.savez_compressed(
    feat_path,
    k3=features_test[3],
    k8=features_test[8],
    k15=features_test[15],
    test_idx=test_idx,
)
print(f'\nSaved: {feat_path}')

k=3: 

test=(68614, 120)  

train=(294851, 120)  zeros=97.5%
k=8: 

test=(68614, 120)  

train=(294851, 120)  zeros=93.3%
k=15: 

test=(68614, 120)  

train=(294851, 120)  zeros=87.5%



Saved: /Users/bridget/Desktop/projects/laloo-sae/output/2A/features_for_paper.npz


### 5. Compute correlations and activation rates → top-feature CSVs

In [7]:
def compute_feature_stats(acts, acts_train, y_train, rmsd, energy, good_mask, top_n=20):
    """
    For each feature column in acts [N, 120]:
      corr_rmsd              Spearman correlation with continuous RMSD
      corr_energy            Spearman correlation with energy
      activation_rate        fraction of poses where feature > 0
      good/bad_activation_rate  activation rate split by 4Å label
      probe_weight           coefficient from L1 logistic regression (bad-pose probe)
      probe_rank             rank by probe_weight (1 = most discriminative)

    `acts`/`rmsd`/`energy`/`good_mask` are the split whose correlations get reported
    (and, when it's the train split, used to pick top_spearman below — see call site).
    `acts_train`/`y_train` always come from the train split; the L1 probe is fit on
    train regardless of which split `acts` is, so probe_weight/probe_rank are never
    fit on data being evaluated.

    Top features = 20 highest positive corr_rmsd (Spearman ranking).
    Probe ranking is included as a second, multivariate lens:
      - Spearman: univariate, continuous RMSD relationship per feature
      - L1 probe: multivariate, jointly discriminative after penalising redundancy
    """
    # Fit L1 probe on train activations
    probe = LogisticRegression(
        penalty='l1', solver='saga', max_iter=2000,
        class_weight='balanced', random_state=42
    )
    # y_train is 1 = bad pose, 0 = good pose
    probe.fit(acts_train, y_train)
    weights = probe.coef_[0]                         # shape [120]
    # rank: feature with highest weight = rank 1
    probe_ranks = len(weights) - np.argsort(np.argsort(weights))

    records = []
    for feat_idx in range(acts.shape[1]):
        v              = acts[:, feat_idx]
        corr_r,   _    = spearmanr(v, rmsd)
        corr_e,   _    = spearmanr(v, energy)
        active         = v > 0
        records.append({
            'feature':              feat_idx,
            'corr_rmsd':            corr_r,
            'corr_energy':          corr_e,
            'activation_rate':      active.mean(),
            'good_activation_rate': active[good_mask].mean()  if good_mask.sum()  > 0 else float('nan'),
            'bad_activation_rate':  active[~good_mask].mean() if (~good_mask).sum() > 0 else float('nan'),
            'probe_weight':         weights[feat_idx],
            'probe_rank':           int(probe_ranks[feat_idx]),
        })
    df  = pd.DataFrame(records)
    top = df.nlargest(top_n, 'corr_rmsd').reset_index(drop=True)
    return df, top, probe


# TRAIN-computed stats: this is what actually selects top_spearman (section 7) — the
# probe is fit on train either way, so this call additionally computes corr_rmsd on
# train, keeping feature *selection* fully out of the test set.
train_feature_dfs = {}
# TEST-computed stats: reported/plotted for descriptive purposes only (e.g. "does this
# train-selected feature's correlation hold up on held-out data?") — never used to pick
# which features or thresholds get used.
all_feature_dfs = {}
probes = {}
for k in K_VALUES:
    print(f'\n--- k={k} ---')
    train_df, _, probe = compute_feature_stats(
        features_train[k], features_train[k], y_train,
        rmsd_train, energy_train, good_mask_train, top_n=20
    )
    train_feature_dfs[k] = train_df
    probes[k] = probe

    all_df, _, _ = compute_feature_stats(
        features_test[k], features_train[k], y_train,
        rmsd_test, energy_test, good_mask, top_n=20
    )
    all_feature_dfs[k] = all_df

    # Top 20 by TRAIN corr_rmsd (the actual selection), reported alongside each
    # feature's TEST corr_rmsd so we can see whether it generalizes.
    top_df = train_df.nlargest(20, 'corr_rmsd')[['feature', 'corr_rmsd', 'probe_weight', 'probe_rank']] \
                      .rename(columns={'corr_rmsd': 'corr_rmsd_train'}) \
                      .reset_index(drop=True)
    top_df = top_df.merge(
        all_df[['feature', 'corr_rmsd', 'activation_rate', 'good_activation_rate', 'bad_activation_rate']]
               .rename(columns={'corr_rmsd': 'corr_rmsd_test'}),
        on='feature', how='left'
    )

    csv_path = os.path.join(OUT_DIR, f'top_features_k{k}.csv')
    top_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    print(top_df[['feature', 'corr_rmsd_train', 'corr_rmsd_test', 'probe_weight', 'probe_rank',
                  'activation_rate', 'good_activation_rate', 'bad_activation_rate']]
          .to_string(index=False, float_format='{:.4f}'.format))

    # Show top 5 by probe weight for comparison (probe itself is train-fit already)
    top_probe = train_df.nlargest(5, 'probe_weight')[['feature', 'probe_weight', 'probe_rank', 'corr_rmsd']]
    print(f'\n  Top 5 by probe weight (k={k}, train):')
    print(top_probe.to_string(index=False, float_format='{:.4f}'.format))


--- k=3 ---


Saved: /Users/bridget/Desktop/projects/laloo-sae/output/2A/top_features_k3.csv
 feature  corr_rmsd_train  corr_rmsd_test  probe_weight  probe_rank  activation_rate  good_activation_rate  bad_activation_rate
       6           0.1939             NaN        2.0000           1           0.0000                0.0000               0.0000
      24           0.1816          0.0088        0.8291           7           0.0002                0.0000               0.0002
      66           0.1747          0.1117        0.4292          24           0.0135                0.0031               0.0153
     109           0.1362         -0.0002        0.7189          11           0.0000                0.0000               0.0000
      43           0.1292         -0.1702       -0.0474          94           0.0418                0.0827               0.0348
      68           0.1195          0.0035        0.9240           5           0.0000                0.0000               0.0000
      32           0.1126

Saved: /Users/bridget/Desktop/projects/laloo-sae/output/2A/top_features_k8.csv
 feature  corr_rmsd_train  corr_rmsd_test  probe_weight  probe_rank  activation_rate  good_activation_rate  bad_activation_rate
      51           0.2559          0.0389        0.6979           7           0.0104                0.0000               0.0121
      50           0.1740          0.0189        0.9444           3           0.0052                0.0009               0.0060
     100           0.1652          0.1120        0.5118          12           0.0140                0.0048               0.0155
      84           0.1551          0.0386        0.2835          25           0.0121                0.0021               0.0138
      36           0.1298          0.0451        0.1263          50           0.0525                0.0405               0.0546
     118           0.1083         -0.0127        0.2408          34           0.0011                0.0029               0.0008
      46           0.1051

Saved: /Users/bridget/Desktop/projects/laloo-sae/output/2A/top_features_k15.csv
 feature  corr_rmsd_train  corr_rmsd_test  probe_weight  probe_rank  activation_rate  good_activation_rate  bad_activation_rate
     119           0.2311          0.0037        3.1717           6           0.0011                0.0000               0.0013
     101           0.1738          0.0571        0.2037          53           0.1346                0.0479               0.1495
     110           0.1671             NaN        0.6089          46           0.0000                0.0000               0.0000
      23           0.1061         -0.0385        0.4098          48           0.0215                0.0221               0.0213
     114           0.1045          0.0799       -0.2604         108           0.0935                0.0376               0.1031
     117           0.0992          0.0125        4.0000           4           0.0002                0.0000               0.0003
     103           0.091

### Good-pose feature analysis

Section 7 only surfaced features that activate on *bad* poses. Reviewer nftm flagged this as
one-sided — the paper's interpretability figures need the mirror: features that activate
preferentially on *good* (native-like) poses.

Same discipline as everywhere else in this notebook: top-5 features per k selected by most
**negative** train Spearman correlation with RMSD (activates more as RMSD gets *smaller*, i.e.
pose gets better), threshold chosen on train to maximize F1 for predicting "good pose" from
activation, applied once to test.

**Output files:**
- `good_pose_features_summary.csv` — per-feature precision/recall/F1 for the good-pose direction
- `threshold_sweep_good_k{k}.png` — activation distributions + threshold sweep, same layout as section 7
- `feature_activations_good_top5.csv` — exact case_id/sample_idx where these features fire, so
  real structures can be pulled for the good-pose interpretability figures nftm asked for
  (mirrors the bad-pose export in section 9)


In [2]:
top_good_spearman = {k: train_feature_dfs[k].nsmallest(5, "corr_rmsd")["feature"].astype(int).tolist()
                      for k in K_VALUES}
print("Top 5 good-pose features by Spearman (train, most negative corr_rmsd):", top_good_spearman)

y_good_test  = good_mask.astype(int)
y_good_train = good_mask_train.astype(int)
baseline_good = y_good_test.mean()
print(f"Baseline good-pose rate (test): {baseline_good:.3f}  ({baseline_good*100:.1f}%)")

best_thresholds_good = {}   # {k: {feat: best_thr}} -- chosen from TRAIN only
good_feature_results = []

for k in K_VALUES:
    acts_train_k = features_train[k]
    acts_test_k  = features_test[k]
    top_feats    = top_good_spearman[k]
    best_thresholds_good[k] = {}

    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    fig.suptitle(f"k={k} — top 5 good-pose features (Spearman ρ with RMSD, selected on train), {RMSD_THRESHOLD}Å threshold",
                 fontsize=12)

    for col, feat in enumerate(top_feats):
        a_train      = acts_train_k[:, feat]
        active_train = a_train > 0
        a_act_train  = a_train[active_train]

        corr_r = train_feature_dfs[k].loc[train_feature_dfs[k]["feature"]==feat, "corr_rmsd"].values[0]

        if active_train.sum() < 10:
            print(f"k={k}, Feature {feat}: too few active train samples ({active_train.sum()}), skipping")
            best_thresholds_good[k][feat] = None
            axes[0, col].axis('off')
            axes[1, col].axis('off')
            continue

        thresholds = np.linspace(a_act_train.min(), np.percentile(a_act_train, 99), 200)
        precisions, recalls, f1s = [], [], []
        for thr in thresholds:
            flagged = a_train >= thr
            if flagged.sum() == 0: continue
            prec = y_good_train[flagged].mean()
            rec  = y_good_train[flagged].sum() / y_good_train.sum()
            f1   = 2*prec*rec / (prec+rec+1e-9)
            precisions.append(prec); recalls.append(rec); f1s.append(f1)

        best_idx = int(np.argmax(f1s))
        best_thr = thresholds[best_idx]
        best_thresholds_good[k][feat] = best_thr

        a_test      = acts_test_k[:, feat]
        active_test = a_test > 0
        a_act_test  = a_test[active_test]
        y_act_test  = y_good_test[active_test]

        test_flagged = a_test >= best_thr
        test_prec = y_good_test[test_flagged].mean() if test_flagged.sum() > 0 else float('nan')
        test_rec  = y_good_test[test_flagged].sum() / y_good_test.sum() if test_flagged.sum() > 0 else float('nan')
        test_f1   = 2*test_prec*test_rec / (test_prec+test_rec+1e-9) if test_flagged.sum() > 0 else float('nan')

        print(f"\nk={k}, Feature {feat} (ρ_train={corr_r:.3f}): "
              f"train active={active_train.sum():,} ({active_train.mean()*100:.1f}%)  "
              f"test active={active_test.sum():,} ({active_test.mean()*100:.1f}%)")
        print(f"  Train-selected thr={best_thr:.3f}  "
              f"(train: prec_good={precisions[best_idx]:.3f} rec_good={recalls[best_idx]:.3f} F1={f1s[best_idx]:.3f})")
        print(f"  Applied to test:   prec_good={test_prec:.3f}  rec_good={test_rec:.3f}  F1={test_f1:.3f}  "
              f"flagged={test_flagged.sum():,}")

        good_feature_results.append({
            'k': k, 'feature': feat, 'corr_rmsd_train': corr_r,
            'train_active_pct': active_train.mean()*100, 'test_active_pct': active_test.mean()*100,
            'threshold': best_thr, 'precision_good_test': test_prec, 'recall_good_test': test_rec,
            'f1_good_test': test_f1, 'n_flagged_test': int(test_flagged.sum()),
        })

        ax = axes[0, col]
        bins = np.linspace(0, a_act_test.max() if len(a_act_test) else 1.0, 60)
        ax.hist(a_act_test[y_act_test==0], bins=bins, density=True, alpha=0.6,
                color="salmon", label=f"bad ({(y_act_test==0).sum():,})")
        ax.hist(a_act_test[y_act_test==1], bins=bins, density=True, alpha=0.6,
                color="mediumseagreen", label=f"good ({(y_act_test==1).sum():,})")
        ax.set_yscale("log")
        ax.set_xlabel("Activation value")
        ax.set_ylabel("Log density")
        ax.set_title(f"Feature {feat}  (ρ_train={corr_r:.3f})\n(TEST distribution, active samples only)")
        ax.legend(fontsize=8)

        ax2 = axes[1, col]
        t = thresholds[:len(f1s)]
        ax2.plot(t, precisions, color="steelblue",  lw=2, label="precision (good)")
        ax2.plot(t, recalls,    color="darkorange", lw=2, label="recall (good)")
        ax2.plot(t, f1s,        color="purple",     lw=2, label="F1")
        ax2.axhline(baseline_good, color="grey", linestyle="--", lw=1,
                    label=f"test baseline {baseline_good:.2f}")
        ax2.axvline(best_thr, color="red", linestyle=":", lw=1.5,
                    label=f"best-F1 thr={best_thr:.2f} (train)")
        ax2.set_xlabel("Activation threshold")
        ax2.set_ylabel("Score")
        ax2.set_title(f"Feature {feat} — TRAIN threshold sweep\ntest@thr: prec={test_prec:.2f} rec={test_rec:.2f}")
        ax2.legend(fontsize=7)
        ax2.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"threshold_sweep_good_k{k}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

good_features_df = pd.DataFrame(good_feature_results)
good_features_df.to_csv(os.path.join(OUT_DIR, "good_pose_features_summary.csv"), index=False)
print(f"\nSaved: {os.path.join(OUT_DIR, 'good_pose_features_summary.csv')}")
print(good_features_df.to_string(index=False, float_format='{:.3f}'.format))


NameError: name 'train_feature_dfs' is not defined

In [9]:
# All poses where top good-pose features activate -- train + test, all k values.
# Use these case_id/sample_idx pairs to pull real structures for the good-pose interpretability
# figures nftm asked for (mirrors the bad-pose export in section 9).

records = []
for k in K_VALUES:
    top_feats  = top_good_spearman[k]
    thresholds = best_thresholds_good[k]

    for split_name, acts, meta_split in [
        ('train', features_train[k], metadata_train),
        ('test',  features_test[k],  metadata_test),
    ]:
        for feat in top_feats:
            thr = thresholds.get(feat)
            if thr is None:
                continue
            act_vals = acts[:, feat]
            firing   = act_vals > 0
            for local_idx in np.where(firing)[0]:
                row = meta_split.iloc[local_idx]
                records.append({
                    'k':            k,
                    'split':        split_name,
                    'feature':      feat,
                    'case_id':      row['case_id'],
                    'sample_idx':   row['sample_idx'],
                    'global_idx':   row['global_idx'],
                    'rmsd':         row['rmsd'],
                    'energy':       row['energy'],
                    'activation':   float(act_vals[local_idx]),
                    'above_thresh': bool(act_vals[local_idx] >= thr),
                    'is_good_pose': bool(row['good_pose']),
                })

good_feature_activations_df = pd.DataFrame(records)
out_path = os.path.join(OUT_DIR, 'feature_activations_good_top5.csv')
good_feature_activations_df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Total rows: {len(good_feature_activations_df):,}")
print()
summary = (good_feature_activations_df
           .groupby(['k', 'feature', 'split', 'above_thresh'])
           .size()
           .rename('n_poses'))
print(summary.to_string())


Saved: /Users/bridget/Desktop/projects/laloo-sae/output/2A/feature_activations_good_top5.csv
Total rows: 595,922

k   feature  split  above_thresh
3   67       test   False               9
                    True             8656
             train  True             9894
    78       test   False               1
                    True               58
             train  False               2
                    True             3428
    97       test   True               64
             train  False             601
                    True             8089
    107      test   False              20
                    True              402
             train  False             744
                    True            11516
    108      test   False               3
                    True              349
             train  False             218
                    True             8634
8   40       test   False             189
                    True            13298
             